In [1]:
import pandas as pd
from urllib.parse import urlsplit, urlunsplit

In [2]:
history = pd.read_csv("datas/browsing_history.csv")
category_map = pd.read_csv("datas/domain_category_map.csv")

print("Browsing history shape:", history.shape)
history.head()

Browsing history shape: (405, 4)


,timestamp,url,title,browser
0,2026-09-11 16:42:13,https://www.google.com/search?q=kick&rlz=1C1RX...,kick - Google Search,Chrome
1,2026-09-11 16:42:15,https://kick.com/,Kick,Chrome
2,2026-09-11 16:42:38,https://kick.com/vaadhiyaar,VAADHIYAAR Stream - Watch Live on Kick,Chrome
3,2026-09-11 16:43:01,https://www.google.com/search?q=zenclass+guvi&...,zenclass guvi - Google Search,Chrome
4,2026-09-11 16:43:03,https://v2.zenclass.in/,GUVI,Chrome


In [6]:
# Check timestamp columns
print(history.columns)

Index(['timestamp', 'url', 'title', 'browser'], dtype='object')


In [7]:
# Find timestamp columns
timestamp_cols = [
    col for col in history.columns
    if col.lower().startswith("timestamp")
]

# Convert timestamp values to datetime
def parse_timestamp(series):
    values = series.dropna().astype(str)

    # Numeric timestamps are treated as Unix seconds
    if len(values) > 0 and values.str.fullmatch(r"\d+(\.\d+)?").all():
        return pd.to_datetime(
            pd.to_numeric(series, errors="coerce"),
            unit="s",
            errors="coerce"
        )

    return pd.to_datetime(series, errors="coerce")

# Select the timestamp column with the most valid parsed dates
parsed_timestamps = {
    col: parse_timestamp(history[col])
    for col in timestamp_cols
}

timestamp_col = max(
    parsed_timestamps,
    key=lambda col: parsed_timestamps[col].notna().sum()
)

history["timestamp"] = parsed_timestamps[timestamp_col]

# Remove rows with invalid timestamps
history = history.dropna(subset=["timestamp"]).copy()

# Convert timezone-aware timestamps to India local time
if history["timestamp"].dt.tz is not None:
    history["timestamp"] = (
        history["timestamp"]
        .dt.tz_convert("Asia/Kolkata")
        .dt.tz_localize(None)
    )

print("Timestamp column used:", timestamp_col)
history[["timestamp"]].head()

Timestamp column used: timestamp


,timestamp
0,2026-09-11 16:42:13
1,2026-09-11 16:42:15
2,2026-09-11 16:42:38
3,2026-09-11 16:43:01
4,2026-09-11 16:43:03


In [8]:
##cleaning url

def clean_url(url):
    if pd.isna(url):
        return None

    parts = urlsplit(str(url))

    return urlunsplit((
        parts.scheme,
        parts.netloc,
        parts.path,
        "",
        ""
    ))

history["url"] = history["url"].apply(clean_url)

history[["url"]].head()

,url
0,https://www.google.com/search
1,https://kick.com/
2,https://kick.com/vaadhiyaar
3,https://www.google.com/search
4,https://v2.zenclass.in/


In [9]:
##  Extracting Domain from URL

def extract_domain(url):
    if pd.isna(url):
        return None

    domain = urlsplit(url).hostname

    if domain:
        domain = domain.lower()

        if domain.startswith("www."):
            domain = domain[4:]

    return domain

history["domain"] = history["url"].apply(extract_domain)

history[["url", "domain"]].head()

,url,domain
0,https://www.google.com/search,google.com
1,https://kick.com/,kick.com
2,https://kick.com/vaadhiyaar,kick.com
3,https://www.google.com/search,google.com
4,https://v2.zenclass.in/,v2.zenclass.in


In [10]:
## Map Domains to Categories

category_map["domain"] = (
    category_map["domain"].str.lower().str.strip()
)

domain_to_category = dict(zip(
    category_map["domain"],
    category_map["category"]
))

history["category"] = history["domain"].map(domain_to_category)

history["category"] = history["category"].fillna("other")

history[["domain", "category"]].head()

,domain,category
0,google.com,search
1,kick.com,entertainment
2,kick.com,entertainment
3,google.com,search
4,v2.zenclass.in,learning


In [11]:
## Create Time-Based Features

history["hour"] = history["timestamp"].dt.hour
history["date"] = history["timestamp"].dt.strftime("%Y-%m-%d")
history["day_name"] = history["timestamp"].dt.day_name()

history[["timestamp", "hour", "date", "day_name"]].head()

,timestamp,hour,date,day_name
0,2026-09-11 16:42:13,16,2026-09-11,Friday
1,2026-09-11 16:42:15,16,2026-09-11,Friday
2,2026-09-11 16:42:38,16,2026-09-11,Friday
3,2026-09-11 16:43:01,16,2026-09-11,Friday
4,2026-09-11 16:43:03,16,2026-09-11,Friday


In [25]:
# Add browser column if it is missing
if "browser" not in history.columns:
    history["browser"] = "chrome"

history["browser"] = history["browser"].fillna("chrome").str.lower()

# Add title column if it is missing
if "title" not in history.columns:
    history["title"] = ""

# Select the required columns
clean_history = history[
    [
        "timestamp",
        "url",
        "domain",
        "category",
        "browser"
    ]
].copy()

# Save timestamp in ISO format
clean_history["timestamp"] = (
    clean_history["timestamp"]
    .dt.strftime("%Y-%m-%dT%H:%M:%S")
)

# Save the cleaned file
clean_history.to_csv(
    "datas/clean_browsing_history.csv",
    index=False
)


clean_history.head()

,timestamp,url,domain,category,browser
0,2026-09-11T16:42:13,https://www.google.com/search,google.com,search,chrome
1,2026-09-11T16:42:15,https://kick.com/,kick.com,entertainment,chrome
2,2026-09-11T16:42:38,https://kick.com/vaadhiyaar,kick.com,entertainment,chrome
3,2026-09-11T16:43:01,https://www.google.com/search,google.com,search,chrome
4,2026-09-11T16:43:03,https://v2.zenclass.in/,v2.zenclass.in,learning,chrome


Sessionization

In [28]:
df = pd.read_csv("datas/clean_browsing_history.csv")

In [31]:
df

,timestamp,url,domain,category,browser
0,2026-09-11T16:42:13,https://www.google.com/search,google.com,search,chrome
1,2026-09-11T16:42:15,https://kick.com/,kick.com,entertainment,chrome
2,2026-09-11T16:42:38,https://kick.com/vaadhiyaar,kick.com,entertainment,chrome
3,2026-09-11T16:43:01,https://www.google.com/search,google.com,search,chrome
4,2026-09-11T16:43:03,https://v2.zenclass.in/,v2.zenclass.in,learning,chrome
...,...,...,...,...,...
400,2026-09-13T19:51:40,https://drive.google.com/drive/folders/1OE3b4i...,drive.google.com,productivity,chrome
401,2026-09-13T19:51:54,https://drive.google.com/drive/folders/1OE3b4i...,drive.google.com,productivity,chrome
402,2026-09-13T19:51:54,https://drive.google.com/drive/folders/1OE3b4i...,drive.google.com,productivity,chrome
403,2026-09-13T19:52:04,https://drive.google.com/drive/folders/1OE3b4i...,drive.google.com,productivity,chrome


Sessionization

In [36]:
import pandas as pd

df = pd.read_csv("datas/clean_browsing_history.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])

df = df.sort_values("timestamp").reset_index(drop=True)

df.head()

,timestamp,url,domain,category,browser
0,2026-09-11 16:42:13,https://www.google.com/search,google.com,search,chrome
1,2026-09-11 16:42:15,https://kick.com/,kick.com,entertainment,chrome
2,2026-09-11 16:42:38,https://kick.com/vaadhiyaar,kick.com,entertainment,chrome
3,2026-09-11 16:43:01,https://www.google.com/search,google.com,search,chrome
4,2026-09-11 16:43:03,https://v2.zenclass.in/,v2.zenclass.in,learning,chrome


In [37]:
##create sessions

# Calculate the time gap between consecutive visits
df["time_gap"] = df["timestamp"].diff()

# Start a new session if the gap is more than 15 minutes
df["new_session"] = df["time_gap"] > pd.Timedelta(minutes=15)

# The first visit starts the first session
df.loc[0, "new_session"] = True

# Assign session IDs
df["session_id"] = df["new_session"].cumsum()

df[["timestamp", "domain", "category", "session_id"]].head(10)

,timestamp,domain,category,session_id
0,2026-09-11 16:42:13,google.com,search,1
1,2026-09-11 16:42:15,kick.com,entertainment,1
2,2026-09-11 16:42:38,kick.com,entertainment,1
3,2026-09-11 16:43:01,google.com,search,1
4,2026-09-11 16:43:03,v2.zenclass.in,learning,1
5,2026-09-11 16:43:03,v2.zenclass.in,learning,1
6,2026-09-11 16:43:04,v2.zenclass.in,learning,1
7,2026-09-11 16:43:04,v2.zenclass.in,learning,1
8,2026-09-11 16:43:09,v2.zenclass.in,learning,1
9,2026-09-11 16:43:09,v2.zenclass.in,learning,1


In [40]:
##Calculate session summary statistics

# Calculate basic statistics for each session
session_summary = df.groupby("session_id").agg(
    start_time=("timestamp", "min"),
    end_time=("timestamp", "max"),
    visit_count=("timestamp", "count"),
    unique_domains=("domain", "nunique"),
    unique_categories=("category", "nunique")
).reset_index()

# Calculate session duration in minutes
session_summary["duration_minutes"] = (
    session_summary["end_time"] - session_summary["start_time"]
).dt.total_seconds() / 60

session_summary.head()

,session_id,start_time,end_time,visit_count,unique_domains,unique_categories,duration_minutes
0,1,2026-09-11 16:42:13,2026-09-11 16:52:56,23,6,5,10.716667
1,2,2026-09-11 17:20:25,2026-09-11 17:30:11,5,2,2,9.766667
2,3,2026-09-11 17:45:42,2026-09-11 18:59:29,68,3,3,73.783333
3,4,2026-09-11 19:37:53,2026-09-11 19:38:37,9,1,1,0.733333
4,5,2026-09-11 20:29:08,2026-09-11 20:53:11,44,2,2,24.050000


In [41]:
##Calculate category counts and ratios

# Count visits by category in each session
category_counts = pd.crosstab(
    df["session_id"],
    df["category"]
)

# Rename columns to make them clear
category_counts.columns = [
    category + "_count"
    for category in category_counts.columns
]

# Calculate category ratios
category_ratios = category_counts.div(
    category_counts.sum(axis=1),
    axis=0
)

category_ratios.columns = [
    category.replace("_count", "_ratio")
    for category in category_ratios.columns
]

# Combine counts and ratios
category_stats = pd.concat(
    [category_counts, category_ratios],
    axis=1
).reset_index()

session_summary = session_summary.merge(
    category_stats,
    on="session_id",
    how="left"
)

session_summary.head()

,session_id,start_time,end_time,visit_count,unique_domains,unique_categories,duration_minutes,entertainment_count,learning_count,productivity_count,researching_count,search_count,shopping_count,entertainment_ratio,learning_ratio,productivity_ratio,researching_ratio,search_ratio,shopping_ratio
0,1,2026-09-11 16:42:13,2026-09-11 16:52:56,23,6,5,10.716667,6,8,4,0,4,1,0.26087,0.347826,0.173913,0.000000,0.173913,0.043478
1,2,2026-09-11 17:20:25,2026-09-11 17:30:11,5,2,2,9.766667,4,0,0,0,1,0,0.80000,0.000000,0.000000,0.000000,0.200000,0.000000
2,3,2026-09-11 17:45:42,2026-09-11 18:59:29,68,3,3,73.783333,0,60,0,3,5,0,0.00000,0.882353,0.000000,0.044118,0.073529,0.000000
3,4,2026-09-11 19:37:53,2026-09-11 19:38:37,9,1,1,0.733333,0,0,0,9,0,0,0.00000,0.000000,0.000000,1.000000,0.000000,0.000000
4,5,2026-09-11 20:29:08,2026-09-11 20:53:11,44,2,2,24.050000,0,43,0,0,1,0,0.00000,0.977273,0.000000,0.000000,0.022727,0.000000


In [42]:
##Calculate category switching behavior

# Compare each visit's category with the previous visit's category
df["previous_category"] = df.groupby("session_id")["category"].shift(1)

# Mark category changes
df["category_switch"] = (
    df["category"] != df["previous_category"]
) & df["previous_category"].notna()

# Count category switches in each session
switch_counts = df.groupby("session_id")["category_switch"].sum()

switch_counts = switch_counts.reset_index(
    name="category_switches"
)

# Calculate switching ratio
session_summary = session_summary.merge(
    switch_counts,
    on="session_id",
    how="left"
)

session_summary["switching_ratio"] = (
    session_summary["category_switches"]
    / (session_summary["visit_count"] - 1).clip(lower=1)
)

session_summary.head()

,session_id,start_time,end_time,visit_count,unique_domains,unique_categories,duration_minutes,entertainment_count,learning_count,productivity_count,...,search_count,shopping_count,entertainment_ratio,learning_ratio,productivity_ratio,researching_ratio,search_ratio,shopping_ratio,category_switches,switching_ratio
0,1,2026-09-11 16:42:13,2026-09-11 16:52:56,23,6,5,10.716667,6,8,4,...,4,1,0.26087,0.347826,0.173913,0.000000,0.173913,0.043478,7,0.318182
1,2,2026-09-11 17:20:25,2026-09-11 17:30:11,5,2,2,9.766667,4,0,0,...,1,0,0.80000,0.000000,0.000000,0.000000,0.200000,0.000000,1,0.250000
2,3,2026-09-11 17:45:42,2026-09-11 18:59:29,68,3,3,73.783333,0,60,0,...,5,0,0.00000,0.882353,0.000000,0.044118,0.073529,0.000000,12,0.179104
3,4,2026-09-11 19:37:53,2026-09-11 19:38:37,9,1,1,0.733333,0,0,0,...,0,0,0.00000,0.000000,0.000000,1.000000,0.000000,0.000000,0,0.000000
4,5,2026-09-11 20:29:08,2026-09-11 20:53:11,44,2,2,24.050000,0,43,0,...,1,0,0.00000,0.977273,0.000000,0.000000,0.022727,0.000000,1,0.023256


In [ ]:
##Save sessionized browsing data

# Save browsing history with session IDs
df.to_csv(
    "datas/sessionized_browsing_history.csv",
    index=False
)

# Save session-level summary
session_summary.to_csv(
    "datas/session_summary.csv",
    index=False
)

session_summary.head()

Sessionized browsing history saved.
Session summary saved.


,session_id,start_time,end_time,visit_count,unique_domains,unique_categories,duration_minutes,entertainment_count,learning_count,productivity_count,...,search_count,shopping_count,entertainment_ratio,learning_ratio,productivity_ratio,researching_ratio,search_ratio,shopping_ratio,category_switches,switching_ratio
0,1,2026-09-11 16:42:13,2026-09-11 16:52:56,23,6,5,10.716667,6,8,4,...,4,1,0.26087,0.347826,0.173913,0.000000,0.173913,0.043478,7,0.318182
1,2,2026-09-11 17:20:25,2026-09-11 17:30:11,5,2,2,9.766667,4,0,0,...,1,0,0.80000,0.000000,0.000000,0.000000,0.200000,0.000000,1,0.250000
2,3,2026-09-11 17:45:42,2026-09-11 18:59:29,68,3,3,73.783333,0,60,0,...,5,0,0.00000,0.882353,0.000000,0.044118,0.073529,0.000000,12,0.179104
3,4,2026-09-11 19:37:53,2026-09-11 19:38:37,9,1,1,0.733333,0,0,0,...,0,0,0.00000,0.000000,0.000000,1.000000,0.000000,0.000000,0,0.000000
4,5,2026-09-11 20:29:08,2026-09-11 20:53:11,44,2,2,24.050000,0,43,0,...,1,0,0.00000,0.977273,0.000000,0.000000,0.022727,0.000000,1,0.023256


RAM Correlation (Time Alignment)


In [44]:
import pandas as pd

history = pd.read_csv("datas/sessionized_browsing_history.csv")
ram = pd.read_csv("datas/ram_log.csv")

# Convert timestamps to datetime
history["timestamp"] = pd.to_datetime(history["timestamp"])
ram["timestamp"] = pd.to_datetime(ram["timestamp"])

print("Browsing data:", history.shape)
print("RAM data:", ram.shape)

history.head()

Browsing data: (405, 10)
RAM data: (1522, 4)


,timestamp,url,domain,category,browser,time_gap,new_session,session_id,previous_category,category_switch
0,2026-09-11 16:42:13,https://www.google.com/search,google.com,search,chrome,NaN,True,1,NaN,False
1,2026-09-11 16:42:15,https://kick.com/,kick.com,entertainment,chrome,0 days 00:00:02,False,1,search,True
2,2026-09-11 16:42:38,https://kick.com/vaadhiyaar,kick.com,entertainment,chrome,0 days 00:00:23,False,1,entertainment,False
3,2026-09-11 16:43:01,https://www.google.com/search,google.com,search,chrome,0 days 00:00:23,False,1,entertainment,True
4,2026-09-11 16:43:03,https://v2.zenclass.in/,v2.zenclass.in,learning,chrome,0 days 00:00:02,False,1,search,True


In [45]:
##clean ram data

ram_columns = [
    "ram_used_mb",
    "ram_available_mb",
    "browser_ram_mb"
]

for column in ram_columns:
    ram[column] = pd.to_numeric(ram[column], errors="coerce")

ram = ram.dropna(subset=["timestamp"] + ram_columns)

ram = ram[
    (ram["ram_used_mb"] >= 0) &
    (ram["ram_available_mb"] >= 0) &
    (ram["browser_ram_mb"] >= 0)
].copy()

ram = ram.drop_duplicates(subset=["timestamp"])

ram = ram.sort_values("timestamp").reset_index(drop=True)

ram.head()

,timestamp,ram_used_mb,ram_available_mb,browser_ram_mb
0,2026-09-11 16:41:42,6711.39,791.07,1481.47
1,2026-09-11 16:41:50,6755.12,747.35,1548.62
2,2026-09-11 16:41:57,6562.60,939.87,1030.46
3,2026-09-11 16:42:05,5669.52,1832.95,865.22
4,2026-09-11 16:42:12,5855.55,1646.91,1108.21


In [46]:
##Match browsing events with RAM logs,nearest match within 10 seconds

# Sort both datasets by timestamp
history = history.sort_values("timestamp").reset_index(drop=True)
ram = ram.sort_values("timestamp").reset_index(drop=True)

# Match each browsing event with the nearest RAM measurement
matched_data = pd.merge_asof(
    history,
    ram,
    on="timestamp",
    direction="nearest",
    tolerance=pd.Timedelta(seconds=10)
)

matched_data.head()

,timestamp,url,domain,category,browser,time_gap,new_session,session_id,previous_category,category_switch,ram_used_mb,ram_available_mb,browser_ram_mb
0,2026-09-11 16:42:13,https://www.google.com/search,google.com,search,chrome,NaN,True,1,NaN,False,5855.55,1646.91,1108.21
1,2026-09-11 16:42:15,https://kick.com/,kick.com,entertainment,chrome,0 days 00:00:02,False,1,search,True,5855.55,1646.91,1108.21
2,2026-09-11 16:42:38,https://kick.com/vaadhiyaar,kick.com,entertainment,chrome,0 days 00:00:23,False,1,entertainment,False,6340.24,1162.22,1922.53
3,2026-09-11 16:43:01,https://www.google.com/search,google.com,search,chrome,0 days 00:00:23,False,1,entertainment,True,6463.62,1038.84,2170.11
4,2026-09-11 16:43:03,https://v2.zenclass.in/,v2.zenclass.in,learning,chrome,0 days 00:00:02,False,1,search,True,6453.28,1049.19,2223.93


In [47]:
##Check the matching results

matched_count = matched_data["ram_used_mb"].notna().sum()
unmatched_count = matched_data["ram_used_mb"].isna().sum()

print("Total browsing events:", len(matched_data))
print("Matched events:", matched_count)
print("Unmatched events:", unmatched_count)

Total browsing events: 405
Matched events: 188
Unmatched events: 217


In [59]:
##Calculate RAM usage percentage

total_ram = (
    matched_data["ram_used_mb"] +
    matched_data["ram_available_mb"]
)

matched_data["ram_percent"] = (
    matched_data["ram_used_mb"] / total_ram * 100
).where(total_ram > 0)

matched_data.head()

,timestamp,url,domain,category,browser,time_gap,new_session,session_id,previous_category,category_switch,ram_used_mb,ram_available_mb,browser_ram_mb,ram_percent,time_gap_minutes
0,2026-09-11 16:42:13,https://www.google.com/search,google.com,search,chrome,NaT,True,1,NaN,False,5855.55,1646.91,1108.21,78.048400,NaN
1,2026-09-11 16:42:15,https://kick.com/,kick.com,entertainment,chrome,0 days 00:00:02,False,1,search,True,5855.55,1646.91,1108.21,78.048400,0.03
2,2026-09-11 16:42:38,https://kick.com/vaadhiyaar,kick.com,entertainment,chrome,0 days 00:00:23,False,1,entertainment,False,6340.24,1162.22,1922.53,84.508814,0.38
3,2026-09-11 16:43:01,https://www.google.com/search,google.com,search,chrome,0 days 00:00:23,False,1,entertainment,True,6463.62,1038.84,2170.11,86.153342,0.38
4,2026-09-11 16:43:03,https://v2.zenclass.in/,v2.zenclass.in,learning,chrome,0 days 00:00:02,False,1,search,True,6453.28,1049.19,2223.93,86.015406,0.03


In [60]:
matched_data.drop("time_gap_minutes",axis=1,inplace=True)

In [ ]:
# Convert time_gap to timedelta, then convert it to minutes
matched_data["time_gap"] = (
    pd.to_timedelta(matched_data["time_gap"], errors="coerce")
    .dt.total_seconds() / 60
)


In [62]:
matched_data.rename(
    columns={"time_gap": "time_gap_min"},
    inplace=True
)

In [63]:
matched_data["time_gap_min"] = matched_data["time_gap_min"].round(2)

matched_data.head()

,timestamp,url,domain,category,browser,time_gap_min,new_session,session_id,previous_category,category_switch,ram_used_mb,ram_available_mb,browser_ram_mb,ram_percent
0,2026-09-11 16:42:13,https://www.google.com/search,google.com,search,chrome,NaN,True,1,NaN,False,5855.55,1646.91,1108.21,78.048400
1,2026-09-11 16:42:15,https://kick.com/,kick.com,entertainment,chrome,0.03,False,1,search,True,5855.55,1646.91,1108.21,78.048400
2,2026-09-11 16:42:38,https://kick.com/vaadhiyaar,kick.com,entertainment,chrome,0.38,False,1,entertainment,False,6340.24,1162.22,1922.53,84.508814
3,2026-09-11 16:43:01,https://www.google.com/search,google.com,search,chrome,0.38,False,1,entertainment,True,6463.62,1038.84,2170.11,86.153342
4,2026-09-11 16:43:03,https://v2.zenclass.in/,v2.zenclass.in,learning,chrome,0.03,False,1,search,True,6453.28,1049.19,2223.93,86.015406


In [ ]:
##RAM statistics for each session

session_ram_stats = matched_data.groupby("session_id").agg(
    visit_count=("timestamp", "count"),
    mean_ram_used_mb=("ram_used_mb", "mean"),
    peak_ram_used_mb=("ram_used_mb", "max"),
    mean_browser_ram_mb=("browser_ram_mb", "mean"),
    peak_browser_ram_mb=("browser_ram_mb", "max"),
    mean_ram_percent=("ram_percent", "mean")
).reset_index()

session_ram_stats.head()

,session_id,visit_count,mean_ram_used_mb,peak_ram_used_mb,mean_browser_ram_mb,peak_browser_ram_mb,mean_ram_percent
0,1,23,6486.186087,7139.35,2264.601739,2970.12,86.454064
1,2,5,NaN,NaN,NaN,NaN,NaN
2,3,68,6173.978864,6494.58,2050.136591,2341.89,82.292674
3,4,9,NaN,NaN,NaN,NaN,NaN
4,5,44,NaN,NaN,NaN,NaN,NaN


In [ ]:
ram_data = matched_data.dropna(
    subset=["ram_used_mb", "browser_ram_mb", "ram_percent"]
).copy()

ram_data


In [50]:
## RAM statistics for each category

category_ram_stats = matched_data.groupby("category").agg(
    visit_count=("timestamp", "count"),
    mean_ram_used_mb=("ram_used_mb", "mean"),
    peak_ram_used_mb=("ram_used_mb", "max"),
    mean_browser_ram_mb=("browser_ram_mb", "mean"),
    peak_browser_ram_mb=("browser_ram_mb", "max"),
    mean_ram_percent=("ram_percent", "mean")
).reset_index()

category_ram_stats.head()

,category,visit_count,mean_ram_used_mb,peak_ram_used_mb,mean_browser_ram_mb,peak_browser_ram_mb,mean_ram_percent
0,entertainment,143,6396.649286,6657.55,2231.161429,2970.12,85.260651
1,learning,188,6288.963197,7139.35,2268.054754,3282.60,83.825292
2,productivity,13,6245.382308,6523.77,1808.666154,2359.36,83.244383
3,researching,19,6201.210000,6968.01,1783.390000,2270.12,82.655632
4,search,33,6027.968500,6769.98,1709.533500,2588.14,80.346532


In [64]:
# Save RAM statistics for each category
category_ram_stats.to_csv(
    "datas/category_ram_stats.csv",
    index=False
)
